# Supervised pathology feature preparation


In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
while not (PROJECT_ROOT / "modeling_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this code from the MMDLPC_Code_PDF folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parent / "MMDLPC_Code_PDF_Data"
OUTPUT_DIR = PROJECT_ROOT / '03_Pathomics/00_Preprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_INPUT_DIR = Path(os.environ.get('MMDLPC_EXTERNAL_INPUTS', str(DATA_ROOT / 'External_Inputs')))


## Supervised


In [ ]:
import re

def id_map(x):
    if x.startswith('TCGA') :
        return x[:12] 
    else:
        items = re.split('[ |\-|_]', x)
        return items[0]

In [ ]:
from collections import namedtuple
import onekey_algo.custom.components as okcomp
from onekey_algo import OnekeyDS as okds

import pandas as pd
import numpy as np

# 读取数据，B超诊断阳性=1，bc_data.csv是要读取的数据。
import pandas as pd
import os
os.makedirs(str(OUTPUT_DIR), exist_ok=True)
os.makedirs(str(OUTPUT_DIR), exist_ok=True)

# 读取数据，B超诊断阳性=1，bc_data.csv是要读取的数据。
prob_histo = pd.read_csv(str(DATA_ROOT / '03_Pathomics/superwise_path_prob_histogram_reference.csv'))
prob_tfidf = pd.read_csv(str(DATA_ROOT / '03_Pathomics/superwise_path_prob_tfidf_reference.csv'))
prob = pd.merge(prob_histo, prob_tfidf, on='ID', how='inner', suffixes=['_histo', '_tfidf'])
prob['ID'] = prob['ID'].astype(str)

pred_histo = pd.read_csv(str(DATA_ROOT / '03_Pathomics/superwise_path_pred_histogram_reference.csv'))
pred_tfidf = pd.read_csv(str(DATA_ROOT / '03_Pathomics/superwise_path_pred_tfidf_reference.csv'))
pred = pd.merge(pred_histo, pred_tfidf, on='ID', how='inner', suffixes=['_histo', '_tfidf'])
pred['ID'] = pred['ID'].astype(str)

features = pd.merge(prob, pred, on='ID', how='inner')
features['ID'] = features['ID'].map(id_map)
analysis_ids = pd.read_csv(DATA_ROOT / '00_Shared_Data_and_Code/Data/P_fixed_partition.csv', dtype={'ID': str})['ID']
features = features[features['ID'].isin(analysis_ids)].copy()
features.to_csv(str(OUTPUT_DIR / 'Supervised_path_features.csv'), index=False, header=True)
labels = ['HRR_ANY']
featrues_not_use = ['ID']

label_data = pd.read_csv(str(DATA_ROOT / '00_Shared_Data_and_Code/Data/CPGEA-TCGA 20230106 OK.csv'), header=0)[['ID', labels[0], 'group']]
label_data['ID'] = label_data['ID'].astype(str)
structed_data = pd.merge(features, label_data, left_on='ID', right_on='ID', how='inner')
structed_data

In [ ]:
sf = ['ID','prob-0.52', 'prob-0.56', 'prob-0.57', 'prob-0.58', 'prob-0.59',
       'prob-0.6', 'prob-0.62', 'prob-0.64', 'prob-0.65', 'prob-0.66',
       'prob-0.68', 'prob-0.7', 'prob-0.72', 'prob-0.73', 'prob-0.74',
       'prob-0.75', 'prob-0.77', 'prob-0.78', 'prob-0.81', 'prob-0.82',
       'prob-0.84', 'prob-0.85', 'prob-0.86', 'prob-0.87', 'prob-0.88',
       'prob-0.89', 'prob-0.91', 'prob-0.93', 'prob-0.94', 'prob-0.95',
       'prob-0.97', 'prob-0.98', 'prob-0.99', 'prob05', 'prob051', 'prob053',
       'prob054', 'prob067', 'prob079', 'prob099', 'prob10', 'pred1']
structed_data[sf].to_csv(str(OUTPUT_DIR / 'Supervised_path_sel_features.csv'), index=False)

In [ ]:
# 删掉ID这一列。
ids = structed_data['ID']
structed_data = structed_data.drop(['ID'], axis=1)
structed_data.columns

In [ ]:
structed_data.describe()

In [ ]:
from onekey_algo.custom.components.comp1 import normalize_df
# data = normalize_df(structed_data, not_norm=labels, group='group')
# data = data.dropna(axis=1)
data = structed_data
data.describe()

In [ ]:
# 如果需要选择相关系数使用对应的相关系数即可。
# pearson_corr = data.corr('pearson')
# kendall_corr = data.corr('kendall')
spearman_corr = data.corr('spearman')

import seaborn as sns
import matplotlib.pyplot as plt
from onekey_algo.custom.components.comp1 import draw_matrix
# plt.figure(figsize=(50.0, 40.0))

# # 选择可视化的相关系数
# draw_matrix(spearman_corr, annot=True, cmap='YlGnBu', cbar=False)
# plt.show()

In [ ]:
from onekey_algo.custom.components.comp1 import select_feature
sel_feature = select_feature(spearman_corr, topn=1, verbose=True)
sel_feature

In [ ]:
sel_data = data[sel_feature + ['group']]
sel_data.describe()

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp

group_info = 'group'
n_classes = 2
train_data = sel_data[(sel_data[group_info] != 'CPGEA')]
train_ids = ids[train_data.index]
train_data = train_data.reset_index()
train_data = train_data.drop('index', axis=1)
y_data = train_data[labels]
X_data = train_data.drop(labels + [group_info], axis=1)

test_data = sel_data[sel_data[group_info] == 'CPGEA']
test_ids = ids[test_data.index]
test_data = test_data.reset_index()
test_data = test_data.drop('index', axis=1)
y_test_data = test_data[labels]
X_test_data = test_data.drop(labels + [group_info], axis=1)

y_all_data = sel_data[labels]
X_all_data = sel_data.drop(labels + [group_info], axis=1)

column_names = X_data.columns
print(f"训练集样本数：{X_data.shape}, 测试集样本数：{X_test_data.shape}")

In [ ]:
# X_data = X_data[selected_features[0]]
# X_test_data = X_test_data[selected_features[0]]
X_data.columns